# Multimodal RAG with the Gemini API File Search Tool

> Build a **fully managed** multimodal RAG pipeline that retrieves across **text + images** using Google's `gemini-embedding-2` and the **File Search** tool — no vector DB to host, no embedding pipeline to maintain.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1x63wXZt9mgVPIWpgY8qswy-Z_Ouc_O6P?usp=sharing)

## What You'll Learn

1. **Create** a File Search Store with multimodal embeddings
2. **Upload** PDFs and images into a unified retrieval index
3. **Query** the store via Gemini with grounded citations
4. **Inspect citations** — pull back text passages *and* the actual cited images
5. **Filter with metadata** for category/season/price-tier scoping
6. **Return structured output** (Pydantic schemas) from a RAG response
7. **Tune chunking** for long documents
8. **Manage stores** — list, delete documents, delete stores

## Architecture

```
PDFs + Images ──▶ File Search Store (gemini-embedding-2)
                          │
                          ▼
User Query ──▶ Gemini + file_search tool ──▶ Retrieved chunks + media
                          │
                          ▼
                Grounded answer + citations (text & images)
```

## Why File Search?

- **Fully managed** — no vector DB to provision, no embedding pipeline to maintain
- **Free storage and free query embeddings** — pay only at index time + standard context tokens
- **Native multimodal** — images embedded directly (no OCR), even images *inside* PDFs
- **Built-in citations** — every claim links back to a chunk, page, or image

## Prerequisites

- Python 3.9+
- A Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey)
- A few PDFs / images to index (we'll use sample files — swap with your own)

---

**Part of [Awesome GenAI Toolkit](https://github.com/shubh-vedi/awesome-genai-toolkit)** — Star the repo if this helps you!

📖 Based on the official guide: [Multimodal RAG with the Gemini API File Search tool](https://dev.to/googleai/multimodal-rag-with-the-gemini-api-file-search-tool-a-developer-guide-5878)

## Step 0: Install Dependencies

In [14]:
!pip install -qU google-genai pydantic

## Step 1: Setup & Configuration

Authenticate with the Gemini API. In Colab, store your key in **Secrets** (🔑 sidebar) as `GOOGLE_API_KEY` — the SDK picks it up automatically.

In [15]:
import os
import time

# Option 1: Set your API key here (for quick testing — NEVER commit a real key!)
# os.environ["GOOGLE_API_KEY"] = "..."

# Option 2: Use Google Colab secrets (recommended)
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass  # Not running in Colab

from google import genai
from google.genai import types

client = genai.Client()

MODEL = "gemini-3-flash-preview"           # generation model
EMBEDDING_MODEL = "models/gemini-embedding-2"  # multimodal embeddings

print("Setup complete!")

Setup complete!


## Step 2: Grab Sample Files

We'll build a tiny **product catalog** — one PDF spec sheet plus a few sneaker images. Replace these URLs with your own files for a real project.

In [16]:
import urllib.request
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Sample PDF — replace with your own product catalog / docs
sample_pdf_url = "https://arxiv.org/pdf/2312.10997.pdf"  # RAG survey paper as a stand-in
pdf_path = data_dir / "product_catalog.pdf"
if not pdf_path.exists():
    print("Downloading sample PDF...")
    urllib.request.urlretrieve(sample_pdf_url, pdf_path)

# Sample images — Picsum returns random photos, fine for a demo.
# In production, swap these for real product photography.
image_urls = {
    "sneaker_red.png":   "https://picsum.photos/seed/red/512",
    "sneaker_blue.jpeg": "https://picsum.photos/seed/blue/512",
    "sneaker_white.png": "https://picsum.photos/seed/white/512",
}
for name, url in image_urls.items():
    p = data_dir / name
    if not p.exists():
        urllib.request.urlretrieve(url, p)

print("Files ready:", [p.name for p in data_dir.iterdir()])

Files ready: ['sneaker_red.png', 'sneaker_blue.jpeg', 'sneaker_white.png', 'product_catalog.pdf']


## Step 3: Create a File Search Store

The store is a managed embedding index. Pick `gemini-embedding-2` for multimodal — it can embed images directly (and images *inside* PDFs).

**Embedding model choices:**
- `gemini-embedding-001` — text-only, cheaper
- `gemini-embedding-2` — text **+ images**, what we want here

In [17]:
file_search_store = client.file_search_stores.create(
    config={
        "display_name": "product-catalog",
        "embedding_model": EMBEDDING_MODEL,
    }
)
print(f"Created store: {file_search_store.name}")

Created store: fileSearchStores/productcatalog-cfiyk0q817gi


## Step 4: Upload Documents and Images

Uploads are **asynchronous operations** — the API chunks, embeds, and indexes in the background. We poll `operations.get()` until each one is `done`.

With `gemini-embedding-2`, image-only files *and* images embedded inside PDFs are both indexed for visual retrieval.

In [18]:
def upload_and_wait(path: Path, display_name: str | None = None, **extra_config):
    """Upload a file to the store and block until indexing finishes."""
    config = {"display_name": display_name or path.name, **extra_config}
    op = client.file_search_stores.upload_to_file_search_store(
        file_search_store_name=file_search_store.name,
        file=str(path),
        config=config,
    )
    while not op.done:
        time.sleep(5)
        op = client.operations.get(op)
    return op

# 1. Upload the PDF
print("Uploading PDF...")
upload_and_wait(pdf_path, display_name="Product Catalog")

# 2. Upload the images
for name in image_urls:
    print(f"Uploading {name}...")
    upload_and_wait(data_dir / name)

print("All files indexed!")

Uploading PDF...
Uploading sneaker_red.png...
Uploading sneaker_blue.jpeg...
Uploading sneaker_white.png...
All files indexed!


## Step 5: Query the Store with the `file_search` Tool

Pass the store name in the `tools` config — Gemini will retrieve the most relevant chunks (text **or** image) and generate a grounded answer.

> ⚠️ **About the demo data:** The sample PDF here is a *RAG survey paper*, not a real product catalog, and the images are random Picsum photos. So we'll ask a question the indexed PDF can actually answer. **When you swap in your own files, change this query** — if nothing in the store matches, the model falls back to general knowledge and `grounding_metadata` comes back `None` (no citations).

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    # Aligned with the indexed PDF (a RAG survey paper). Swap for your own
    # query when you swap in your own data.
    contents="What are the main components of a Retrieval-Augmented Generation pipeline according to the paper?",
    config={
        "tools": [{
            "file_search": {
                "file_search_store_names": [file_search_store.name]
            }
        }]
    },
)

print(response.text)

## Step 6: Handle Citations — Text *and* Images

Every grounded response carries `grounding_metadata`:
- `grounding_chunks` — the source passages or media used
- `grounding_supports` — which **claim** in the answer maps to which chunks

Image citations expose a `media_id` you can download to show the user.

In [ ]:
grounding = response.candidates[0].grounding_metadata

cited_dir = Path("cited")
cited_dir.mkdir(exist_ok=True)

if grounding is None:
    # The model didn't invoke the file_search tool — usually because nothing
    # in the store was relevant to the query. The answer is then ungrounded.
    print("⚠️  No grounding metadata: the model answered from general knowledge.")
    print("   Try a query that matches your indexed content, or upload more relevant files.")
else:
    for chunk in grounding.grounding_chunks:
        ctx = chunk.retrieved_context
        if ctx.media_id:
            print(f"🖼️  Cited image: {ctx.title}")
            print(f"     Media ID: {ctx.media_id}")
            blob = client.file_search_stores.download_media(media_id=ctx.media_id)
            out = cited_dir / f"cited_{ctx.title}"
            with open(out, "wb") as f:
                f.write(blob)
            print(f"     Saved to {out}")
        else:
            print(f"📄 Cited text: {ctx.title}")
            if ctx.page_number:
                print(f"     Page: {ctx.page_number}")
            print(f"     {ctx.text[:200]}...")
        print()

    print("--- Claim → chunk mapping ---")
    for support in grounding.grounding_supports or []:
        print(f"Claim: '{support.segment.text}'")
        print(f"  Grounded in chunks: {support.grounding_chunk_indices}")

## Step 7: Custom Metadata + Filtered Retrieval

Attach metadata at upload time, then narrow retrieval at query time. This is how you scope a single store across **categories**, **seasons**, **tenants**, **price tiers**, etc.

Supported value types: `string_value`, `numeric_value`.

In [ ]:
# Re-upload the PDF with extra metadata so we can filter by it.
upload_and_wait(
    pdf_path,
    display_name="Spring 2026 Shoes",
    custom_metadata=[
        {"key": "category",   "string_value": "footwear"},
        {"key": "season",     "string_value": "spring-2026"},
        {"key": "price_tier", "numeric_value": 2},
    ],
)

filtered = client.models.generate_content(
    model=MODEL,
    contents="Do you have blue spring shoes?",
    config={
        "tools": [{
            "file_search": {
                "file_search_store_names": [file_search_store.name],
                "metadata_filter": 'category="footwear" AND season="spring-2026"',
            }
        }]
    },
)
print(filtered.text)

## Step 8: Structured Output (Pydantic)

Combine `file_search` with `response_schema` to get **typed JSON** back instead of free-form prose — perfect for downstream UI / pipelines.

In [ ]:
from pydantic import BaseModel, Field

class ProductMatch(BaseModel):
    name: str = Field(description="Product name")
    description: str = Field(description="Brief product description")
    confidence: str = Field(description="How confident the match is")

structured = client.models.generate_content(
    model=MODEL,
    contents="Find products similar to a red running shoe",
    config={
        "tools": [{
            "file_search": {
                "file_search_store_names": [file_search_store.name]
            }
        }],
        "response_mime_type": "application/json",
        "response_schema": ProductMatch.model_json_schema(),
    },
)
print(structured.text)

## Step 9: Tune Chunking for Long Documents

Default chunking is fine for most docs, but for **long technical manuals** you may want smaller chunks with overlap to improve recall on dense passages.

In [25]:
upload_and_wait(
    pdf_path,
    display_name="Technical Manual",
    chunking_config={
        "white_space_config": {
            "max_tokens_per_chunk": 200,
            "max_overlap_tokens": 20,
        }
    },
)
print("Re-indexed with custom chunking.")

Re-indexed with custom chunking.


## Step 10: Store Management

List your stores, inspect documents, and clean up when you're done. **Stores persist across sessions** until you explicitly delete them, so housekeeping matters if you're iterating.

In [26]:
# List all stores
print("--- Stores ---")
for store in client.file_search_stores.list():
    print(f"{store.name} — {store.display_name}")

# List documents in our store
print("\n--- Documents in our store ---")
for doc in client.file_search_stores.documents.list(parent=file_search_store.name):
    print(f"  {doc.name}")

--- Stores ---
fileSearchStores/productcatalog-fa885wmwtwpf — product-catalog
fileSearchStores/productcatalog-bbgucslcuf7k — product-catalog
fileSearchStores/productcatalog-cfiyk0q817gi — product-catalog

--- Documents in our store ---
  fileSearchStores/productcatalog-cfiyk0q817gi/documents/product-catalog-uthskgj5op84
  fileSearchStores/productcatalog-cfiyk0q817gi/documents/sneakerredpng-6ogyclnoa3bg
  fileSearchStores/productcatalog-cfiyk0q817gi/documents/sneakerbluejpeg-t00a816kmfwi
  fileSearchStores/productcatalog-cfiyk0q817gi/documents/sneakerwhitepng-2st2ix8sg2vn
  fileSearchStores/productcatalog-cfiyk0q817gi/documents/spring-2026-shoes-rg82yt5nhvp4
  fileSearchStores/productcatalog-cfiyk0q817gi/documents/technical-manual-9ahvcpglnto6


## Where to Take This Next

**Use cases this pattern fits well:**
- 🛒 **Visual product search** — match a query against image + spec sheets in one store
- 🔬 **Research docs** — retrieve charts and diagrams alongside paper text
- 🧾 **Insurance claims** — combine submitted forms with damage photos
- 🎨 **Design systems** — searchable by visual appearance
- 🏠 **Real estate** — floor plans + interior photography

**Pricing recap (at the time of writing):**
- Indexing: pay once for embeddings
- Storage: **free**
- Query embeddings: **free**
- Retrieved tokens: standard context-token charges

**Further reading:**
- [File Search documentation](https://ai.google.dev/gemini-api/docs/file-search)
- [Official quickstart notebook](https://github.com/google-gemini/cookbook/blob/main/quickstarts/File_Search.ipynb)
- [`google-genai` Python SDK](https://github.com/googleapis/python-genai)

---

⭐ **If this helped, star [Awesome GenAI Toolkit](https://github.com/shubh-vedi/awesome-genai-toolkit)!**